# 07 — Reimagine Kontext Test

Diagnostic notebook for `FluxKontextPipeline` (image + prompt editing, no ControlNet, no Gradio).

**Pipeline:** `FluxKontextPipeline` — `black-forest-labs/FLUX.1-Kontext-dev`

**Three test prompts** are run against the same fixed source image with distinct seeds:
- Prompt 1 seed 1001 — building as industrial beast
- Prompt 2 seed 2002 — street buried in bureaucratic infrastructure
- Prompt 3 seed 3003 — animal elites vs human masses

**Then repeated** with Satirefic LoRA active (if file is found).

**Two hashes per image:**
- `pixel_hash` — SHA-256 of raw pixel bytes before PNG encoding (what the model produced in RAM)
- `file_hash` — SHA-256 of the saved .png file (what landed on disk)

**Filenames:** `07_<label>_seed<N>_<uuid8>.png`

Do not edit `05_Cartoonify_Gradio_Unified.ipynb` until these tests pass.

---

## 1 — Confirm A100 GPU

In [ ]:
!nvidia-smi


## 2 — Install Dependencies

In [ ]:
!pip install --quiet diffusers transformers accelerate sentencepiece safetensors
!pip install --quiet huggingface_hub


In [ ]:
import os
os.kill(os.getpid(), 9)
print('IGNORE: session crashed — continue to the next cell.')


## 3 — Imports

In [ ]:
import gc
import hashlib
import os
import uuid
import warnings
import torch
from PIL import Image, ImageDraw
from diffusers import FluxKontextPipeline
from safetensors import safe_open

print(f'torch : {torch.__version__}')
print(f'CUDA  : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')


## 4 — Mount Google Drive

In [ ]:
import shutil
from google.colab import drive

if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    os.system('fusermount -u /content/drive 2>/dev/null || true')
    shutil.rmtree('/content/drive', ignore_errors=True)

drive.mount('/content/drive')


## 5 — API Tokens

Accept the [FLUX.1-Kontext-dev licence](https://huggingface.co/black-forest-labs/FLUX.1-Kontext-dev)
on HuggingFace before running — it is gated separately from FLUX.1-dev.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=False)
print('Logged in to Hugging Face')


## 6 — Configuration

In [ ]:
BASE_MODEL = 'black-forest-labs/FLUX.1-Kontext-dev'

LORA_SATIREFIC_PATH         = '/content/drive/MyDrive/satirefic/satirefic/satirefic.safetensors'
LORA_SATIREFIC_ADAPTER_NAME = 'satirefic'

# Set this to use your own image. Leave None to auto-find or create a test image.
INPUT_IMAGE_OVERRIDE = None

OUTPUT_DIR    = '/content/drive/MyDrive/cartoonify/outputs/07_kontext_test'
OUTPUT_DIR_05 = '/content/drive/MyDrive/cartoonify/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Exact values from Notebook 05 cell-config
GLOBAL_PROMPT_PREFIX = 'crude black and white newspaper caricature'

SATIREFIC_STYLE_BOOST = (
    'maximum grotesque political satire, extreme caricature exaggeration, '
    'absurd anti-authoritarian newspaper cartoon, rough dirty ink linework, '
    'ugly distorted faces, pig-politician symbolism, visual irony, '
    'anarchist press style, flat white paper, brutal black ink, no readable text'
)

# Three test prompts — deliberately different subject matter and spatial focus
KONTEXT_SPECS = [
    {
        'id':    1,
        'seed':  1001,
        'label': 'p1_industrial_beast',
        'story': 'turn the building into a hungry industrial beast devouring the street',
        'desc':  'Building as industrial beast',
    },
    {
        'id':    2,
        'seed':  2002,
        'label': 'p2_tax_cables',
        'story': 'cover the street with tax papers, cables, and cooling pipes',
        'desc':  'Street buried in bureaucratic infrastructure',
    },
    {
        'id':    3,
        'seed':  3003,
        'label': 'p3_animal_elites',
        'story': (
            'transform powerful figures into grotesque animal elites '
            'while ordinary people remain human'
        ),
        'desc':  'Animal elites vs human masses',
    },
]

print(f'Base model : {BASE_MODEL}')
print(f'Output dir : {OUTPUT_DIR}')
for spec in KONTEXT_SPECS:
    print(f"  [{spec['id']}] seed={spec['seed']}  {spec['desc']}")


## Setup — Load Source Image

Priority order:
1. `INPUT_IMAGE_OVERRIDE` path if set
2. Most recent PNG from the Notebook 05 output directory
3. Programmatic test image (urban street scene)

In [ ]:
from IPython.display import display as ipy_display


def create_urban_test_image(size=1024):
    """Programmatic placeholder: sky, buildings, street, stick figures."""
    img  = Image.new('RGB', (size, size), (135, 150, 170))
    draw = ImageDraw.Draw(img)
    draw.rectangle([0, size * 55 // 100, size, size], fill=(90, 82, 70))
    buildings = [
        (30,  180, 380, (78, 78, 100)),
        (230, 220, 520, (68, 68, 90)),
        (510, 190, 440, (74, 74, 96)),
        (750, 240, 350, (82, 82, 104)),
    ]
    ground_y = size * 55 // 100
    for bx, bw, bh, color in buildings:
        top = ground_y - bh
        draw.rectangle([bx, top, bx + bw, ground_y], fill=color)
        for row in range(2, bh // 50):
            for col in range(1, bw // 40):
                wx = bx + col * 36 + 4
                wy = top + row * 48
                draw.rectangle([wx, wy, wx + 22, wy + 28], fill=(195, 208, 228))
    for fx in [160, 380, 580, 820]:
        fy = ground_y + 55
        draw.ellipse([fx - 14, fy - 28, fx + 14, fy], fill=(210, 175, 135))
        draw.line([fx, fy, fx, fy + 75], fill=(55, 55, 75), width=5)
        draw.line([fx, fy + 28, fx - 28, fy + 56], fill=(55, 55, 75), width=4)
        draw.line([fx, fy + 28, fx + 28, fy + 56], fill=(55, 55, 75), width=4)
        draw.line([fx, fy + 75, fx - 18, fy + 120], fill=(55, 55, 75), width=4)
        draw.line([fx, fy + 75, fx + 18, fy + 120], fill=(55, 55, 75), width=4)
    return img


src_image = None
src_label = ''

if INPUT_IMAGE_OVERRIDE and os.path.isfile(INPUT_IMAGE_OVERRIDE):
    src_image = Image.open(INPUT_IMAGE_OVERRIDE).convert('RGB')
    src_label = INPUT_IMAGE_OVERRIDE
    print(f'Loaded user image: {src_label}')
elif os.path.isdir(OUTPUT_DIR_05):
    candidates = sorted([f for f in os.listdir(OUTPUT_DIR_05) if f.endswith('.png')])
    if candidates:
        src_path  = os.path.join(OUTPUT_DIR_05, candidates[-1])
        src_image = Image.open(src_path).convert('RGB')
        src_label = src_path
        print(f'Loaded NB05 output: {src_label}')
    else:
        src_image = create_urban_test_image()
        src_label = 'programmatic test image'
        print('No NB05 outputs found — created programmatic test image')
else:
    src_image = create_urban_test_image()
    src_label = 'programmatic test image'
    print('NB05 output dir not found — created programmatic test image')

src_image = src_image.resize((1024, 1024), Image.LANCZOS)
src_save  = os.path.join(OUTPUT_DIR, 'source_image.png')
src_image.save(src_save)
print(f'Source size : {src_image.size}')
print(f'Saved to    : {src_save}')
print()
ipy_display(src_image.resize((512, 512)))


## Setup — Load Model

Loads `FluxKontextPipeline` — takes image+prompt, no ControlNet.

⏱️ First download: ~24 GB. Warm cache: ~90 seconds.

In [ ]:
gc.collect()
torch.cuda.empty_cache()
_free, _total = torch.cuda.mem_get_info()
print(f'GPU before load: {_free/1e9:.1f} GB free / {_total/1e9:.1f} GB total')
print()

print(f'Loading FluxKontextPipeline from {BASE_MODEL} ...')
pipe = FluxKontextPipeline.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16).to('cuda')

_free2, _ = torch.cuda.mem_get_info()
print(f'GPU after load : {_free2/1e9:.1f} GB free (used {(_free - _free2)/1e9:.1f} GB)')
print(f'Pipeline class : {type(pipe).__name__}')
print('FluxKontextPipeline ready')


## Generate — Helpers

`sha256_pixels` hashes raw pixel bytes from the in-memory PIL object **before** PNG encoding.  
`sha256_file` hashes the saved file **after** PNG encoding.  
Filenames include task label, seed, and a UUID so no two runs can overwrite each other.

In [ ]:
def sha256_pixels(img):
    """SHA-256 of raw RGB pixel bytes — what the model produced in memory."""
    h = hashlib.sha256()
    h.update(img.tobytes())
    return h.hexdigest()


def sha256_file(path):
    """SHA-256 of the saved PNG file on disk."""
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        h.update(f.read())
    return h.hexdigest()


def build_final_prompt(story_prompt, use_satirefic=False):
    """Replicates build_final_prompt() from Notebook 05 cell-inference."""
    base = story_prompt.strip()
    if not base.startswith(GLOBAL_PROMPT_PREFIX):
        base = GLOBAL_PROMPT_PREFIX + ', ' + base
    if use_satirefic and SATIREFIC_STYLE_BOOST not in base:
        base = base + ', ' + SATIREFIC_STYLE_BOOST
    return base


def run_kontext(source, prompt, seed, label, guidance=2.5, steps=28):
    """
    Single FluxKontextPipeline call with an isolated generator.
    Returns (PIL image, saved path, pixel_hash, file_hash).

    pixel_hash: SHA-256 of img.tobytes() BEFORE save.
                If this matches across calls the model produced the same output.
    file_hash:  SHA-256 of the .png file AFTER save.
                If pixel_hash differs but file_hash matches, a filename collision overwrote a file.
    """
    gen = torch.Generator(device='cuda').manual_seed(seed)

    print(f'  seed             : {seed}')
    print(f'  generator device : {gen.device}')
    print(f'  guidance         : {guidance}  steps: {steps}')
    print(f'  prompt[:100]     : {prompt[:100]}...')

    with torch.inference_mode():
        out = pipe(
            image=source,
            prompt=prompt,
            width=1024,
            height=1024,
            num_inference_steps=steps,
            guidance_scale=guidance,
            generator=gen,
            joint_attention_kwargs={'scale': 1.0},
            max_sequence_length=512,
        )

    img      = out.images[0]                    # PIL image from this pipeline call
    px_hash  = sha256_pixels(img)               # hash pixels BEFORE encoding

    uid      = uuid.uuid4().hex[:8]
    filename = f'07_{label}_seed{seed}_{uid}.png'
    path     = os.path.join(OUTPUT_DIR, filename)
    img.save(path)
    f_hash   = sha256_file(path)                # hash file AFTER encoding

    print(f'  filename         : {filename}')
    print(f'  pixel SHA-256    : {px_hash}  <- model output')
    print(f'  file  SHA-256    : {f_hash}  <- on disk')
    return img, path, px_hash, f_hash


## Test 1 — Three Prompts, No LoRA

Runs all three KONTEXT_SPECS against the same source image. All three **pixel hashes** must differ.

In [ ]:
nolora_results = []

# No LoRA has been loaded at this point — disable_lora() is not called here.
# LoRA loading happens in cell-23-load-lora, which runs after these tests.
print('=== Test 1: Three Prompts, No LoRA ===')
print('Expected: all pixel hashes differ.')
print()

for spec in KONTEXT_SPECS:
    final_prompt = build_final_prompt(spec['story'], use_satirefic=False)
    print(f"--- Prompt {spec['id']}: {spec['desc']} (seed {spec['seed']}) ---")
    print(f'  full prompt: {final_prompt[:120]}...')
    img, path, px_h, f_h = run_kontext(
        src_image, final_prompt, spec['seed'],
        f"nolora_p{spec['id']}_{spec['label']}",
    )
    nolora_results.append({
        'id':         spec['id'],
        'desc':       spec['desc'],
        'seed':       spec['seed'],
        'img':        img,
        'path':       path,
        'pixel_hash': px_h,
        'file_hash':  f_h,
    })
    print()

print('=== No-LoRA Results ===')
for r in nolora_results:
    print(f"  Prompt {r['id']} seed {r['seed']:5d} | pixel: {r['pixel_hash'][:24]}... | file: {r['file_hash'][:24]}...")

unique_px = len(set(r['pixel_hash'] for r in nolora_results))
n = len(nolora_results)
if unique_px == n:
    print()
    print(f'PASS: all {n} prompts produced distinct pixel hashes')
else:
    print()
    print(f'FAIL: only {unique_px} distinct pixel hashes from {n} prompts')

    seen_px, seen_f = {}, {}
    for r in nolora_results:
        if r['pixel_hash'] in seen_px:
            print(f"  Prompt {r['id']} pixel_hash == Prompt {seen_px[r['pixel_hash']]}  => model produced same output")
        seen_px[r['pixel_hash']] = r['id']
        if r['file_hash'] in seen_f:
            print(f"  Prompt {r['id']} file_hash  == Prompt {seen_f[r['file_hash']]}  => same bytes on disk (filename collision)")
        seen_f[r['file_hash']] = r['id']

    print()
    print('=== DEBUG: every variable passed to pipe() ===')
    print(f'  type(pipe)               : {type(pipe).__name__}')
    print(f'  image (source)           : same PIL object for every call')
    print(f'  width / height           : 1024 x 1024')
    print(f'  num_inference_steps      : 28')
    print(f'  guidance_scale           : 2.5')
    print(f'  generator                : fresh torch.Generator("cuda").manual_seed(seed) each call')
    print(f'  joint_attention_kwargs   : {{"scale": 1.0}}')
    print(f'  max_sequence_length      : 512')
    raise AssertionError('Three-prompt test FAILED — review DEBUG output above before continuing.')


## Verify Outputs — No LoRA

In [ ]:
def tile_images(images, thumb=400):
    n = len(images)
    canvas = Image.new('RGB', (thumb * n, thumb), (20, 20, 24))
    for i, img in enumerate(images):
        canvas.paste(img.resize((thumb, thumb), Image.LANCZOS), (i * thumb, 0))
    return canvas


print('Source image:')
ipy_display(src_image.resize((512, 512)))
print()
print('Test 1 — three prompts, no LoRA (left=beast, centre=cables, right=animals):')
ipy_display(tile_images([r['img'] for r in nolora_results]))


## Load LoRA — Inspect Satirefic for Kontext Compatibility

FLUX.1-Kontext-dev shares the base transformer architecture with FLUX.1-dev but adds image
conditioning layers. A LoRA trained on FLUX.1-dev may load with unexpected-key warnings;
the Kontext-specific layers are untouched.

In [ ]:
lora_exists = os.path.isfile(LORA_SATIREFIC_PATH)
print(f'Path   : {LORA_SATIREFIC_PATH}')
print(f'Exists : {lora_exists}')
print()

if lora_exists:
    with safe_open(LORA_SATIREFIC_PATH, framework='pt', device='cpu') as f:
        metadata = f.metadata()
        all_keys = list(f.keys())

    print('--- Metadata ---')
    if metadata:
        for k, v in metadata.items():
            print(f'  {k}: {v}')
    else:
        print('  (no metadata stored in file)')

    print(f'\n--- Keys: first 30 of {len(all_keys)} total ---')
    for k in all_keys[:30]:
        print(f'  {k}')

    flux_keys    = [k for k in all_keys if any(p in k for p in ['transformer', 'lora_transformer'])]
    unet_keys    = [k for k in all_keys if 'unet' in k.lower()]
    kontext_keys = [k for k in all_keys if 'kontext' in k.lower()]
    lora_a       = [k for k in all_keys if 'lora_A' in k or 'lora_down' in k]

    print()
    print('--- Compatibility with FLUX.1-Kontext-dev ---')
    print(f'  Total keys          : {len(all_keys)}')
    print(f'  FLUX-pattern keys   : {len(flux_keys)}')
    print(f'  UNet-pattern keys   : {len(unet_keys)}')
    print(f'  Kontext-specific    : {len(kontext_keys)}')
    print(f'  LoRA A/down keys    : {len(lora_a)}')

    if len(kontext_keys) > 0:
        print()
        print('  => LoRA includes Kontext-specific keys: trained for FLUX.1-Kontext-dev')
    elif len(flux_keys) > 0 and len(unet_keys) == 0:
        print()
        print('  => FLUX-format LoRA, no Kontext keys: trained on FLUX.1-dev')
        print('     May load with unexpected-key warnings; Kontext conditioning unaffected.')
    elif len(unet_keys) > 0:
        print()
        print('  => UNet keys detected: NOT compatible with FLUX.1-Kontext-dev')
    else:
        print()
        print('  => Unrecognised format: run the load attempt below and check warnings')
else:
    print('LoRA not found — LoRA test cells will print a skip message and continue.')


In [ ]:
_lora_loaded  = False
_lora_verdict = 'not attempted (file not found)'

if lora_exists:
    try:
        import peft.import_utils as _peft_utils
        import peft.tuners.lora.torchao as _peft_torchao
        _orig_fn = _peft_utils.is_torchao_available
        def _safe_fn():
            try:
                return _orig_fn()
            except ImportError:
                return False
        _peft_utils.is_torchao_available = _safe_fn
        _peft_torchao.is_torchao_available = _safe_fn
        print('PEFT torchao patch applied')

        with warnings.catch_warnings(record=True) as caught_warnings:
            warnings.simplefilter('always')
            pipe.load_lora_weights(
                os.path.dirname(LORA_SATIREFIC_PATH),
                weight_name=os.path.basename(LORA_SATIREFIC_PATH),
                adapter_name=LORA_SATIREFIC_ADAPTER_NAME,
            )

        if caught_warnings:
            print(f'{len(caught_warnings)} warning(s) during LoRA load:')
            for w in caught_warnings[:5]:
                print(f'  [{w.category.__name__}] {str(w.message)[:120]}')
        else:
            print('No warnings during LoRA load')

        pipe.disable_lora()
        _lora_loaded = True

        if any('unexpected' in str(w.message).lower() for w in caught_warnings):
            _lora_verdict = 'loaded with unexpected-key warnings (FLUX.1-dev LoRA on Kontext — base transformer affected)'
        elif any('missing' in str(w.message).lower() for w in caught_warnings):
            _lora_verdict = 'loaded with missing-key warnings'
        else:
            _lora_verdict = 'loaded cleanly'

        print(f'Satirefic LoRA loaded as adapter "{LORA_SATIREFIC_ADAPTER_NAME}"')
        print(f'Verdict: {_lora_verdict}')

    except Exception as exc:
        _lora_verdict = f'FAILED: {exc}'
        print(f'ERROR: {_lora_verdict}')
        print('Continuing without LoRA.')
else:
    print('Satirefic LoRA not found — skipping load.')


## Test 2 — Three Prompts, Satirefic LoRA

Same three prompts and seeds with the Satirefic boost appended. Skipped if the LoRA did not load.

In [ ]:
lora_results = []

if _lora_loaded:
    pipe.set_adapters([LORA_SATIREFIC_ADAPTER_NAME], adapter_weights=[0.9])
    pipe.enable_lora()
    print('Satirefic LoRA activated at strength 0.9')
    print()

    for spec in KONTEXT_SPECS:
        final_prompt = build_final_prompt(spec['story'], use_satirefic=True)
        lora_label   = f"satirefic_p{spec['id']}_{spec['label']}"
        print(f"--- Prompt {spec['id']}: {spec['desc']} (seed {spec['seed']}, Satirefic) ---")
        print(f'  full prompt: {final_prompt[:120]}...')
        img, path, px_h, f_h = run_kontext(
            src_image, final_prompt, spec['seed'], lora_label,
        )
        lora_results.append({
            'id':         spec['id'],
            'desc':       spec['desc'],
            'seed':       spec['seed'],
            'img':        img,
            'path':       path,
            'pixel_hash': px_h,
            'file_hash':  f_h,
        })
        print()

    pipe.disable_lora()

    print('=== Satirefic LoRA Results ===')
    for r in lora_results:
        print(f"  Prompt {r['id']} seed {r['seed']:5d} | pixel: {r['pixel_hash'][:24]}... | file: {r['file_hash'][:24]}...")

    print()
    print('--- Pixel hash cross-comparison (no-LoRA vs LoRA) ---')
    for nl, lr in zip(nolora_results, lora_results):
        same   = nl['pixel_hash'] == lr['pixel_hash']
        status = 'WARNING: SAME — LoRA had no visible effect' if same else 'DIFFERENT (expected)'
        print(f"  Prompt {nl['id']} seed {nl['seed']:5d}: {status}")
else:
    print('Satirefic LoRA not loaded — skipping LoRA test.')
    print(f'Place the file at: {LORA_SATIREFIC_PATH}')


## Verify Outputs — With LoRA

In [ ]:
if lora_results:
    print('Test 2 — three prompts, Satirefic LoRA (left=beast, centre=cables, right=animals):')
    ipy_display(tile_images([r['img'] for r in lora_results]))
else:
    print('LoRA test was skipped — no images to display.')


In [ ]:
nolora_distinct = len(set(r['pixel_hash'] for r in nolora_results)) == len(nolora_results)

print('=== Final Summary ===')
print()
print(f'Pipeline                                                 : FluxKontextPipeline')
print(f'Base model                                               : {BASE_MODEL}')
print(f'Source image                                             : {src_label}')
print(f'Satirefic LoRA file found                                : {lora_exists}')
print(f'Satirefic LoRA loaded                                    : {_lora_loaded}')
print(f'Satirefic / Kontext compatibility verdict                : {_lora_verdict}')
print()
print(f'[NO-LORA]  3 prompts -> distinct pixel hashes            : {nolora_distinct}')

if lora_results:
    lora_distinct = len(set(r['pixel_hash'] for r in lora_results)) == len(lora_results)
    lora_differs  = all(nl['pixel_hash'] != lr['pixel_hash'] for nl, lr in zip(nolora_results, lora_results))
    print(f'[LORA]     3 prompts -> distinct pixel hashes            : {lora_distinct}')
    print(f'[LORA]     LoRA output differs from no-LoRA output       : {lora_differs}')

print()
print('No-LoRA pixel hashes (model output):')
for r in nolora_results:
    print(f"  Prompt {r['id']} {r['desc']:45s} seed {r['seed']:5d}: {r['pixel_hash'][:32]}...")

if lora_results:
    print()
    print('Satirefic LoRA pixel hashes (model output):')
    for r in lora_results:
        print(f"  Prompt {r['id']} {r['desc']:45s} seed {r['seed']:5d}: {r['pixel_hash'][:32]}...")

print()
print('=== Compatibility Notes ===')
print('  Satirefic compatibility with FLUX.1-Kontext-dev is unproven until runtime loading')
print('  and generation succeeds. A successful load does not by itself prove a visible')
print('  style effect — the LoRA test images must differ visually from the no-LoRA images.')
print('  A FLUX.1-dev LoRA does not target the Kontext image-conditioning layers;')
print('  style influence may be weaker than on FLUX.1-dev.')
